# Preparación epidemiológica 2024 a nivel de grilla

1. **Objetivo del notebook**

- Limpiar los registros epidemiológicos de 2024 y agregarlos por grilla y semana epidemiológica.

2. **Decisiones metodológicas**

- Asignar cada registro a la celda del centro notificador, usado como proxy espacial.
- Conservar residencia y procedencia sólo como variables descriptivas.
- No completar con cero las combinaciones grilla-semana sin registros.

3. **Fuentes**

- `data/raw/epidemiology/TRANSPARENCIA_final Arica.xlsx`.
- `data/raw/epidemiology/reporting_centers_coordinates.csv`.
- `data/processed/environment/grid_geometry.parquet`, generado por `environmental/01_data_aggregation.ipynb`.

4. **Productos**

- `data/processed/epidemiology/epidemiology_2024_clean.parquet`.
- `data/processed/epidemiology/epidemiology_2024_grid_week.parquet`.


La ubicación del centro notificador es un proxy y no representa residencia, exposición ni lugar de infección.

In [9]:
# Librerías para rutas, tablas, geometrías y normalización de texto
from pathlib import Path
import sys, unicodedata
import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

# Localiza la raíz del proyecto buscando src/paths.py en la carpeta actual y sus padres
for candidate in [Path.cwd(), *Path.cwd().resolve().parents]:
    if (candidate / 'src' / 'paths.py').is_file(): PROJECT_ROOT=candidate; break
else: raise FileNotFoundError('No se encontró src/paths.py.')
sys.path.insert(0,str(PROJECT_ROOT))
# Calendario epidemiológico común y rutas centralizadas del pipeline
from src.epidemiological_calendar import EPI_YEAR,EPI_START,EPI_END,assign_epiweek_2024
from src.paths import EPIDEMIOLOGY_CLEAN_PATH,EPIDEMIOLOGY_GRID_WEEK_PATH,EPIDEMIOLOGY_RAW_PATH,GRID_GEOMETRY_PATH,REPORTING_CENTERS_COORDINATES_PATH
pd.set_option('display.max_columns',None); pd.set_option('display.max_rows',200)


## 1. Carga de las hojas 2023 y 2024 y perfil general

Se leen ambas hojas para cubrir el año epidemiológico 2024. Se muestran las primeras 10 filas y el resumen de columnas.


In [10]:
# Carga las hojas 2023 y 2024 sin agregar columnas
if not EPIDEMIOLOGY_RAW_PATH.is_file():
    raise FileNotFoundError(f'No se encontró el Excel: {EPIDEMIOLOGY_RAW_PATH}')
df_raw = pd.concat(
    pd.read_excel(EPIDEMIOLOGY_RAW_PATH, sheet_name=['2023', '2024']).values(),
    ignore_index=True,
)

display(df_raw.head(10))

# Resume una muestra de valores únicos por columna
def preview(series, max_values=30):
    values = series.dropna().astype('string').drop_duplicates().tolist()
    text = ' | '.join(values[:max_values])
    if len(values) > max_values:
        text += f' | ... ({len(values) - max_values} valores adicionales)'
    return text

column_profile = pd.DataFrame([
    {
        'columna': column,
        'tipo': str(df_raw[column].dtype),
        'nulos': int(df_raw[column].isna().sum()),
        'valores_unicos': int(df_raw[column].nunique(dropna=True)),
        'valores_unicos_muestra': preview(df_raw[column]),
    }
    for column in df_raw.columns
])
display(column_profile)


,SEXO,EDAD,FECHA DE NOTIFICACION,CLASIFICACION DEL CASO,RESULTADO TEST DIAGNOSTICO,LUGAR DE RESIDENCIA,ZONA CENSAL,ANTECEDENTE DE VIAJE,PAIS DE PROCEDENCIA,ZONA DE RIESGO PRIORIZADA PARA EL VECTOR,CENTRO NOTIFICADOR,TIPO DE VIGILANCIA,NOMBRE ZONA RIESGO
0,Hombre,14,2023-11-27,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
1,Hombre,19,2023-11-27,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
2,Hombre,4,2023-11-26,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
3,Hombre,32,2023-11-26,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
4,Hombre,7 días,2023-11-25,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
5,Mujer,55,2023-11-25,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
6,Mujer,1,2023-11-24,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
7,Hombre,9,2023-11-24,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
8,Mujer,26,2023-11-24,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN
9,Hombre,2 meses,2023-11-20,Descartado,Negativo,Arica,No disponible,No,No,No aplica,Centro Centinela,Centinela,NaN


,columna,tipo,nulos,valores_unicos,valores_unicos_muestra
0,SEXO,str,0,3,Hombre | Mujer | mujer
1,EDAD,object,0,96,14 | 19 | 4 | 32 | 7 días | 55 | 1 | 9 | 26 | ...
2,FECHA DE NOTIFICACION,datetime64[us],0,655,2023-11-27 | 2023-11-26 | 2023-11-25 | 2023-11...
3,CLASIFICACION DEL CASO,str,0,5,Descartado | Sospechoso | Confirmado | No conc...
4,RESULTADO TEST DIAGNOSTICO,str,0,5,Negativo | Sin Información | Positivo | Indete...
5,LUGAR DE RESIDENCIA,str,0,5,Arica | Turista | Región Metropolitana | Regió...
6,ZONA CENSAL,str,0,1,No disponible
7,ANTECEDENTE DE VIAJE,str,0,3,No | Si | No
8,PAIS DE PROCEDENCIA,str,0,14,No | Bolivia | Perú | Argentina | Alemania | A...
9,ZONA DE RIESGO PRIORIZADA PARA EL VECTOR,str,0,4,No aplica | No Aplica | No | Si


## 2. Semana epidemiológica

Se conservan solo las fechas del 31-12-2023 al 28-12-2024, correspondientes al año epidemiológico 2024. Se mantienen las columnas originales y solo se agrega `epiweek`.


In [11]:
fechas = pd.to_datetime(df_raw['FECHA DE NOTIFICACION'], dayfirst=True, errors='raise')
df_epi = df_raw.loc[fechas.between(EPI_START, EPI_END)].copy()
df_epi['epiweek'] = assign_epiweek_2024(fechas.loc[df_epi.index])
display(df_epi.head(10))


,SEXO,EDAD,FECHA DE NOTIFICACION,CLASIFICACION DEL CASO,RESULTADO TEST DIAGNOSTICO,LUGAR DE RESIDENCIA,ZONA CENSAL,ANTECEDENTE DE VIAJE,PAIS DE PROCEDENCIA,ZONA DE RIESGO PRIORIZADA PARA EL VECTOR,CENTRO NOTIFICADOR,TIPO DE VIGILANCIA,NOMBRE ZONA RIESGO,epiweek
1123,Hombre,28,2024-01-04,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,1
1124,Hombre,14,2024-01-04,Altamente Probable de Confirmación,Negativo,Arica,No disponible,Si,Perú,No Aplica,Centro Centinela,Centinela,No Aplica,1
1125,Mujer,0,2024-01-03,Altamente Probable de Confirmación,Negativo,Arica,No disponible,Si,Perú,No Aplica,Centro Centinela,Centinela,No Aplica,1
1126,Hombre,10,2024-01-04,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,1
1127,Mujer,28,2024-01-05,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,1
1128,Mujer,25,2024-01-05,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,1
1129,Hombre,1,2024-01-05,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,1
1130,Hombre,20,2024-01-05,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,1
1131,Hombre,77,2024-01-07,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,2
1132,Mujer,10,2024-01-07,Descartado,Negativo,Arica,No disponible,No,No,No Aplica,Centro Centinela,Centinela,No Aplica,2


## 3. Tabla descriptiva de procedencia

Se resume la procedencia de los casos confirmados sin usarla para la asignación espacial.

In [12]:
def normalize_text(series):
    return (series.astype('string').str.strip().str.normalize('NFKD').str.encode('ascii',errors='ignore').str.decode('ascii').str.upper().str.replace(r'\s+',' ',regex=True))

es_confirmado = (
    normalize_text(df_epi['RESULTADO TEST DIAGNOSTICO']).eq('CONFIRMADO')
    | normalize_text(df_epi['CLASIFICACION DEL CASO']).eq('CONFIRMADO')
)

# Tabla descriptiva de casos confirmados y país de procedencia declarado
confirmed_table=df_epi.loc[es_confirmado,['RESULTADO TEST DIAGNOSTICO','PAIS DE PROCEDENCIA']].rename(columns={'RESULTADO TEST DIAGNOSTICO':'RESULTADO TEST','PAIS DE PROCEDENCIA':'PAÍS DE PROCEDENCIA'}).sort_index()
display(confirmed_table)
display(confirmed_table['PAÍS DE PROCEDENCIA'].value_counts(dropna=False).rename_axis('país_de_procedencia').rename('n_casos_confirmados').to_frame())
print('Advertencia metodológica: el centro médico notificador es sólo un proxy espacial del registro. No debe interpretarse como domicilio, exposición ni lugar probable de infección.')


,RESULTADO TEST,PAÍS DE PROCEDENCIA
1210,Confirmado,Bolivia
1447,Confirmado,Perú
1687,Confirmado,Bolivia
2216,Confirmado,Perú
2421,Confirmado,Bolivia
2424,Confirmado,Perú


,n_casos_confirmados
país_de_procedencia,
Bolivia,3
Perú,3


Advertencia metodológica: el centro médico notificador es sólo un proxy espacial del registro. No debe interpretarse como domicilio, exposición ni lugar probable de infección.


De este análisis se desprende que no se detectaron casos autóctonos de dengue en la Región de Arica y Parinacota, por lo que asignar la ubicación del caso a una celda específica de la región no tendría sentido al provenir desde afuera. Por lo mismo, la ubicación del caso será asignada al centro que lo notifica. 

## 4. Coordenadas de centros y asignación a la grilla

Se lee el CSV y se asigna cada centro al interior de una celda mediante una unión espacial. Los centros fuera de la grilla o sobre sus bordes quedan sin celda; las asignaciones múltiples detienen el proceso.


In [13]:
# Lee los centros y la grilla
centers = pd.read_csv(REPORTING_CENTERS_COORDINATES_PATH, usecols=['source_name', 'longitude', 'latitude'])
centers = centers.rename(columns={'source_name': 'CENTRO NOTIFICADOR'})
grid = gpd.read_parquet(GRID_GEOMETRY_PATH)

# Convierte las coordenadas a puntos en el sistema de la grilla
points = gpd.GeoDataFrame(
    centers, geometry=gpd.points_from_xy(centers.longitude, centers.latitude), crs='EPSG:4326'
).to_crs(grid.crs)

# Asigna cada centro a la celda que contiene su punto
center_assignments = gpd.sjoin(points, grid[['grid_id', 'geometry']], how='left', predicate='within')
center_assignments = center_assignments[['CENTRO NOTIFICADOR', 'longitude', 'latitude', 'grid_id']]
if center_assignments['CENTRO NOTIFICADOR'].duplicated().any():
    raise ValueError('Hay centros repetidos o asignados a varias celdas; revisar antes de continuar.')

print('Centros sin celda:', center_assignments.loc[center_assignments.grid_id.isna(), 'CENTRO NOTIFICADOR'].tolist())
display(center_assignments)


Centros sin celda: ['Centro de Salud Familiar E.U. Iris Veliz Hume (Ex Oriente)', 'Clínica San José', 'Hospital Regional Dr. Juan Noé Crevanni (Arica)']


,CENTRO NOTIFICADOR,longitude,latitude,grid_id
0,Centro Centinela,-70.290849,-18.437676,102.0
1,Centro de Salud Familiar E.U. Iris Veliz Hume ...,-70.282440,-18.483935,NaN
2,Centro de Salud Familiar Remigio Sapunar,-70.308152,-18.480391,65.0
3,Clínica San José,-70.306724,-18.483619,NaN
4,Hospital Regional Dr. Juan Noé Crevanni (Arica),-70.313046,-18.482955,NaN
5,Posta de Salud Rural San Miguel de Azapa,-70.178527,-18.518354,330.0
6,SEREMI Salud Arica y Parinacota,-70.315042,-18.479562,65.0


## 5. Unión de asignaciones a los casos confirmados

Se conservan solo los casos confirmados según el criterio de la parte 3 y se unen sus coordenadas por `CENTRO NOTIFICADOR`. `grid_assignment_status` indica si tienen celda (`matched`) o no (`unmatched_center`).


In [14]:
# Conserva solo los casos confirmados y agrega sus coordenadas
cases = df_epi.loc[es_confirmado].assign(
    fecha_notificacion_dt=fechas,
    year=EPI_YEAR,
).merge(
    center_assignments,
    on='CENTRO NOTIFICADOR', how='left', validate='many_to_one',
)
cases['grid_assignment_status'] = np.where(cases['grid_id'].notna(), 'matched', 'unmatched_center')

display(cases[['CENTRO NOTIFICADOR', 'grid_id', 'grid_assignment_status']].head(20))
display(cases['grid_assignment_status'].value_counts(dropna=False).rename('n_registros').to_frame())

,CENTRO NOTIFICADOR,grid_id,grid_assignment_status
0,Centro Centinela,102.0,matched
1,Centro Centinela,102.0,matched
2,SEREMI Salud Arica y Parinacota,65.0,matched
3,Centro Centinela,102.0,matched
4,Clínica San José,NaN,unmatched_center
5,Hospital Regional Dr. Juan Noé Crevanni (Arica),NaN,unmatched_center


,n_registros
grid_assignment_status,
matched,4
unmatched_center,2


## 6. Productos canónicos y auditoría

Se guardan los casos confirmados y su agregado por grilla-semana. `n_cases_confirmed` cuenta los casos confirmados asignados en cada combinación.

El agregado requiere una celda asignada; los casos sin celda se conservan en el archivo individual. Todos los casos pertenecen al periodo epidemiológico 2024.


In [15]:
# Esquema explícito del producto individual limpio
allowed_columns=['RESULTADO TEST DIAGNOSTICO','CLASIFICACION DEL CASO','PAIS DE PROCEDENCIA','LUGAR DE RESIDENCIA','FECHA DE NOTIFICACION','fecha_notificacion_dt','year','epiweek','CENTRO NOTIFICADOR','longitude','latitude','grid_id']
clean_cases=cases[[c for c in allowed_columns if c in cases.columns]].copy()
# El agregado semanal requiere una celda y una semana asignadas
assigned = clean_cases.loc[
    clean_cases.grid_id.notna() & clean_cases.epiweek.notna()
].copy()
# Agregación a una fila por celda y semana epidemiológica
grid_week=assigned.groupby(['grid_id','year','epiweek'],as_index=False).agg(n_cases_confirmed=('epiweek','size'),n_reporting_centers=('CENTRO NOTIFICADOR','nunique'))
# Controles de unicidad, rango y conservación de totales
if int(grid_week.n_cases_confirmed.sum())!=len(assigned): raise ValueError('No se conservaron los casos confirmados.')
# Guardado del producto individual y del agregado grilla–semana
EPIDEMIOLOGY_CLEAN_PATH.parent.mkdir(parents=True,exist_ok=True); EPIDEMIOLOGY_GRID_WEEK_PATH.parent.mkdir(parents=True,exist_ok=True)
clean_cases.to_parquet(EPIDEMIOLOGY_CLEAN_PATH,index=False); grid_week.to_parquet(EPIDEMIOLOGY_GRID_WEEK_PATH,index=False)
print(f'Producto individual: {EPIDEMIOLOGY_CLEAN_PATH} ({len(clean_cases):,} filas)'); print(f'Producto grilla-semana: {EPIDEMIOLOGY_GRID_WEEK_PATH} ({len(grid_week):,} filas)')
display(grid_week.head(20))


Producto individual: C:\tesis_aedes_grilla_2024\data\processed\epidemiology\epidemiology_2024_clean.parquet (6 filas)
Producto grilla-semana: C:\tesis_aedes_grilla_2024\data\processed\epidemiology\epidemiology_2024_grid_week.parquet (4 filas)


,grid_id,year,epiweek,n_cases_confirmed,n_reporting_centers
0,65.0,2024,20,1,1
1,102.0,2024,7,1,1
2,102.0,2024,16,1,1
3,102.0,2024,43,1,1


In [16]:
clean_cases

,RESULTADO TEST DIAGNOSTICO,CLASIFICACION DEL CASO,PAIS DE PROCEDENCIA,LUGAR DE RESIDENCIA,FECHA DE NOTIFICACION,fecha_notificacion_dt,year,epiweek,CENTRO NOTIFICADOR,longitude,latitude,grid_id
0,Confirmado,Confirmado,Bolivia,Arica,2024-02-13,2024-02-13,2024,7,Centro Centinela,-70.290849,-18.437676,102.0
1,Confirmado,Confirmado,Perú,Arica,2024-04-18,2024-04-18,2024,16,Centro Centinela,-70.290849,-18.437676,102.0
2,Confirmado,Confirmado,Bolivia,Turista,2024-05-14,2024-05-14,2024,20,SEREMI Salud Arica y Parinacota,-70.315042,-18.479562,65.0
3,Confirmado,Confirmado,Perú,Región Metropolitana,2024-10-24,2024-10-24,2024,43,Centro Centinela,-70.290849,-18.437676,102.0
4,Confirmado,Confirmado,Bolivia,Arica,2024-02-27,2024-02-27,2024,9,Clínica San José,-70.306724,-18.483619,NaN
5,Confirmado,Confirmado,Perú,Arica,2024-03-04,2024-03-04,2024,10,Hospital Regional Dr. Juan Noé Crevanni (Arica),-70.313046,-18.482955,NaN


## Criterios de cierre

- `grid_id` proviene del centro notificador.
- Los registros de centros sin celda se conservan con `grid_id` faltante.
- El agregado contiene sólo combinaciones grilla-semana observadas. *Haber eliminado los 55 nulos de celdas sin SE asignada, provocó que dos centros quedaran sin celda al corresponder justo con las que ya no forman parte de la grilla. Se opta por dejarlos con Grid_id NaN. 